In [14]:
#pip install srt
#pip install tgt

SyntaxError: invalid syntax (3722349461.py, line 2)

In [28]:
import srt
import os
import tgt
from bisect import bisect_left
import pandas as pd

In [2]:
#почистим OCR: все нг заменим на ӈг, тюркскую ӊ - на ӈ
path = '/Users/air/Downloads/Ненецкие аудио/Чумотека'

for file in os.listdir(path):
    if file.endswith('.srt') and file.endswith('_new.srt') == False:
        with open(os.path.join(path, file), encoding='utf-8') as f:
            subtitle_generator = srt.parse(f.read())
            subtitles = list(subtitle_generator)
            for sub in subtitles:
                sub.content = sub.content.replace(sub.content.rsplit('\n')[-1], '') #в нижней строчке русский текст - отбросим
                sub.content = sub.content.lower().replace('ӊ', 'ӈ') #кое-где распозналась тюркская буква
                sub.content = sub.content.replace('нг', 'ӈг') #перед г только ӈ

        new_file_name = file.replace('.srt', '_new.srt')
        new_file_path = os.path.join(path, new_file_name)

        if not os.path.exists(new_file_path):
            with open(new_file_path, 'w', encoding='utf-8') as f:
                f.write(srt.compose(subtitles))
            print(f"Создан файл {new_file_name}")
        else:
            print(f'Файл {new_file_name} уже существует')

Файл Старик Хув Хоба_new.srt уже существует
Файл Маринзя_new.srt уже существует
Файл Бедняк_new.srt уже существует
Файл Старая журавлиха_new.srt уже существует
Файл Три оленевода Вэра_new.srt уже существует
Создан файл Два Няднгы_new.srt
Создан файл Царь и Поп_new.srt
Файл Три ханты_new.srt уже существует
Файл Ненэй не, парны'я_new.srt уже существует


In [ ]:
with open('/Users/air/Downloads/Ненецкие аудио/Чумотека/Бедняк_new.srt', encoding ='utf-8') as f: #посмотрим что получилось
    subtitle_generator = srt.parse(f.read())
    subtitles = list(subtitle_generator)
            
    for sub in subtitles:
        print(f"Index: {sub.index}")
        print(f"Start: {sub.start}")  # datetime.timedelta object
        print(f"Content: {sub.content}")   

Index: 1
Start: 0:00:09.200000
Content: саванекоця няанд лаханаку вадетам! ваиленяко нюбеӈга.
Index: 2
Start: 0:00:17.480000
Content: ваиленяконд пухцяда таня.
Index: 3
Start: 0:00:23.480000
Content: пухуцяда ӈобтикы луце не
Index: 4
Start: 0:00:26.440000
Content: ваиленякор ялехэ' ханевари мэ'ӈа
Index: 5
Start: 0:00:31.120000
Content: ваиленякор нямзхэй' мэ'ӈа
Index: 6
Start: 0:00:33.679000
Content: сидя ханесэйбциеда
Index: 7
Start: 0:00:35.840000
Content: ӈоб', сидя хоркыця хадаби, халяком' нямзхэнд хадабанакы
Index: 8
Start: 0:00:40.960000
Content: тарем илеӈаха пухуцянда ня
Index: 9
Start: 0:00:45.359000
Content: ваиленякор ся'ны' ӈэбто поӈгацята хая'
Index: 10
Start: 0:00:50.200000
Content: парэӈгода яв' нявна яда
Index: 11
Start: 0:00:53.079000
Content: няби ӈэда хоркада
Index: 12
Start: 0:00:56.039000
Content: ма: ӈэми ӈамгэн' пакалй'?
Index: 13
Start: 0:01:03.160000
Content: ӈэнда пакла'ма сивня сылы'
Index: 14
Start: 0:01:08.679000
Content: ӈэнда си'мы я' ваӈг ӈэвы
Index: 15


In [14]:
tgt_path = '/Users/air/Downloads/Ненецкие аудио/Чумотека/Чумотека tgt'
srt_path = '/Users/air/Downloads/Ненецкие аудио/Чумотека/Чумотека srt'

In [26]:
def nearest_value(sorted_values, x): #находит ближайшее значение из отсортированного списка
    if not sorted_values:
        raise ValueError("Список границ пуст.")
    i = bisect_left(sorted_values, x)
    if i == 0:
        return sorted_values[0]
    if i == len(sorted_values):
        return sorted_values[-1]
    left = sorted_values[i - 1]
    right = sorted_values[i]
    if abs(x - left) <= abs(right - x):
        return left
    return right

In [3]:
def collect_non_empty_boundaries(textgrid, source_tier_name=None): #собирает границы у непустых интервалов
    boundaries = set()
    if source_tier_name is not None:
        tiers = [textgrid.get_tier_by_name(source_tier_name)]
    else:
        tiers = textgrid.tiers
    for tier in tiers:
        if not isinstance(tier, tgt.IntervalTier):
            continue
        for interval in tier.intervals:
            text = str(interval.text).strip()
            if text:
                boundaries.add(float(interval.start_time))
                boundaries.add(float(interval.end_time))
    return sorted(boundaries)

In [24]:
def make_subtitle_intervals(subs, boundaries, xmin, xmax):   #cоздаёт интервалы субтитров, привязанные к ближайшим границам.
    intervals = []
    previous_end = xmin
    for sub in subs:
        raw_start = sub.start.total_seconds()
        raw_end = sub.end.total_seconds()
        label = sub.content
        if not label:
            continue
        start = nearest_value(boundaries, raw_start)
        end = nearest_value(boundaries, raw_end)
        start = max(xmin, min(start, xmax))
        end = max(xmin, min(end, xmax))
        if end <= start:                #если начало и конец совпадают, берём ближайшую следующую границу
            later_boundaries = [b for b in boundaries if b > start]
            if later_boundaries:
                end = later_boundaries[0]
            else:
                continue
        if start < previous_end:         #TextGrid не должен иметь пересекающиеся интервалы
            start = previous_end
        if end <= start:
            continue
        intervals.append(tgt.Interval(start_time=start, end_time=end, text=label))
        previous_end = end
    return intervals

In [27]:
#файлы textgrid размечены точно по голосовой активности, а srt - нет, но промежутки нужной длины. попробуем подвинуть границы 
for file in os.listdir(tgt_path):
    if file.endswith('.TextGrid'):
        #print(file)
        textgrid = tgt.io.read_textgrid(os.path.join(tgt_path, file), include_empty_intervals=True)
        with open(os.path.join(srt_path, file.replace('.TextGrid', '_new.srt')), encoding='utf-8') as f:
            subtitles = srt.parse(f.read())
            subtitles = list(subtitles)
        xmin = textgrid.start_time
        xmax = textgrid.end_time
        boundaries = collect_non_empty_boundaries(textgrid, source_tier_name='текст')
        if not boundaries:
            raise ValueError("Не найдено непустых интервалов в TextGrid.")
        subtitle_intervals = make_subtitle_intervals(subtitles, boundaries, xmin, xmax)
        if not subtitle_intervals:
            raise ValueError("Не получилось создать интервалы субтитров.")
        subtitles_tier = tgt.IntervalTier(xmin, xmax, name='субтитры', objects=subtitle_intervals)
        textgrid.add_tier(subtitles_tier)
        newname = file.replace('.TextGrid', '_new.TextGrid')
        tgt.io.write_to_file(textgrid, os.path.join(tgt_path, newname), format="long" )
        print(f'Создан файл {newname}')

Создан файл Три ханты_new.TextGrid
Создан файл Старик Хув Хоба_new.TextGrid
Создан файл Маринзя_new.TextGrid
Создан файл Ненэй не, парны'я_new.TextGrid
Создан файл Три оленевода Вэра_new.TextGrid
Создан файл Два Няднгы_new.TextGrid
Создан файл Старая журавлиха_new.TextGrid
Создан файл Царь и Поп_new.TextGrid


In [33]:
def pd_from_tgt(path):
    data = []
    for file in os.listdir(path):
        if file.endswith('.TextGrid'):
            tg = tgt.io.read_textgrid(os.path.join(path, file))
            tier = tg.get_tier_by_name('субтитры')
            for i, interval in enumerate(tier):
                    data.append({
                        "file_name" : os.path.join(path.replace('tgt', 'wav'), file.replace('TextGrid', 'wav')),
                        "start_time": interval.start_time,
                        "end_time": interval.end_time,
                        "label": interval.text,
                        "length": interval.end_time - interval.start_time
                    })
    df = pd.DataFrame(data)
    df.to_csv(path.replace(' tgt', '.csv'), index=False)
    return df

In [30]:
def calculate_time(dataframe):
    print(f'Средняя длина интервала {dataframe['length'].mean()}, максимальная {dataframe['length'].max()}, минимальная {dataframe['length'].min()}')
    print(f'Звучащее время {dataframe['length'].sum() // 3600}ч {dataframe['length'].sum() % 3600 // 60}мин {dataframe['length'].sum() % 60}с')

In [46]:
chumoteka_df = pd_from_tgt('/Users/air/Downloads/Ненецкие аудио/Чумотека/Чумотека tgt')
calculate_time(chumoteka_df)

Средняя длина интервала 5.578273401174428, максимальная 10.916076971112204, минимальная 1.012499999999818
Звучащее время 1.0ч 58.0мин 43.455133299745285с
